In [3]:
import matplotlib.pyplot as plt
import numpy as np
from tfim_simulator import simulator_functions

In [4]:
SIZE = 8
LAYERS = SIZE
g_VALUE = 0.5
BOUNDS = 'open'

loss_gsim = simulator_functions(SIZE, LAYERS, g_VALUE,
                                bounds=BOUNDS, return_grad=False)

num_params = (2 * SIZE - 1) * SIZE
params = np.random.uniform(-np.pi, np.pi, num_params)
print(loss_gsim(params))

0.8329652691762929


## Reference Loss

In [1]:
import tequila as tq

def ref_loss(thetas):
    # generate HVA ansatz circuit
    U = tq.QCircuit()
    U += tq.gates.H(np.arange(SIZE)) # such that rho_in is plus state
    for l in range(LAYERS):
        for i in range(SIZE):
            U += tq.gates.Rx(2*tq.Variable(f"beta{l}_{i}"), i)
        for i in range(SIZE - 1):
            U += tq.gates.ExpPauli(tq.PauliString({int(i): "Z", int(i+1): "Z"}), angle=2 * tq.Variable(f"gamma{l}_{i}"))
        if BOUNDS == 'periodic':
            U += tq.gates.ExpPauli(tq.PauliString({int(SIZE-1): "Z", int(0): "Z"}), angle=2 * tq.Variable(f"gamma{l}_{SIZE-1}"))

    # generate TFIM measurement operator
    paulis = []
    for i in range(SIZE):
        paulis.append(tq.PauliString({int(i): "X"}, -g_VALUE))
    for i in range(SIZE-1):
        paulis.append(tq.PauliString({int(i): "Z", int(i+1): "Z"}, 1))
    if BOUNDS == 'periodic':
        paulis.append(tq.PauliString({int(SIZE-1): "Z", int(0): "Z"}, 1))

    H = tq.QubitHamiltonian.from_paulistrings(paulis)

    # define expectation value
    E = tq.ExpectationValue(U, H)

    vars = U.extract_variables()
    init = {var: theta for var, theta in zip(vars, thetas)}

    return tq.simulate(E, init)

In [5]:
print(ref_loss(params))

0.8329650839042024
